## Create a simple UI in Streamlit without using built-in layout functions, manually structuring components using HTML and CSS, and ensuring responsiveness across different screen sizes.

In [1]:
%pip install streamlit pandas scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import streamlit
import pandas
import sklearn
import joblib

print("All libraries installed successfully")

All libraries installed successfully


In [3]:
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("heart.csv")
df = df.drop_duplicates().dropna()

X = df.drop(columns=["target"])
y = df["target"]

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X, y)

joblib.dump(model, "heart_model_v20261013_160903.joblib")

print("Model saved successfully!")

Model saved successfully!


In [4]:
%cd "30oct folder ka complete path"
!streamlit run app.py

[WinError 2] The system cannot find the file specified: '30oct folder ka complete path'
C:\Users\Admin\heart_disease\30oct
^C


In [ ]:
!streamlit run app.py

## Task 1: Implement user input handling for prediction by validating inputs dynamically, handling edge cases such as missing or incorrect values, and optimizing input processing for real-time predictions.

### Step 1: `input.py` (validation + prediction + batch)

In [1]:
%%writefile input.py
"""
HeartCare - user input handling for prediction.

1. Dynamic validation : rules (allowed categories, number ranges) heart.csv se khud banti hain.
2. Edge cases         : missing, text, out-of-range, unknown category, inf/NaN, extra keys, wrong length.
3. Real-time speed    : ek hi predict_proba pass, repeated input ka cache, batch me vectorized validation.
"""
import numpy as np
import pandas as pd
from functools import lru_cache

FEATURES = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
            "thalach", "exang", "oldpeak", "slope", "ca", "thal"]
MISSING_WORDS = {"", "na", "n/a", "nan", "null", "none", "?", "-", "unknown", "not known"}
MAX_MISSING = 3           


def build_rules(df):
    """Training data se rules: <= 5 alag values wala column = category, baaki = number."""
    rules = {}
    for f in FEATURES:
        col = df[f].astype(float).dropna()
        vals = np.sort(col.unique())
        r = {"min": float(col.min()), "max": float(col.max())}
        if len(vals) <= 5:
            r.update(kind="category", allowed=vals.tolist(), fill=float(col.mode()[0]))
        else:
            span = col.max() - col.min()
            lo, hi = col.min() - 0.25 * span, col.max() + 0.25 * span    
            if col.min() >= 0:
                lo = max(lo, 0.0)                                         
            r.update(kind="number", hard_lo=float(lo), hard_hi=float(hi), fill=float(col.median()))
        rules[f] = r
    return rules


def load_rules(csv_path="heart.csv"):
    return build_rules(pd.read_csv(csv_path).drop_duplicates().dropna())


def is_missing(v):
    if v is None or v is pd.NA:
        return True
    if isinstance(v, str):
        return v.strip().lower() in MISSING_WORDS
    return isinstance(v, (float, np.floating)) and np.isnan(v)


def to_number(v):
    if isinstance(v, (bool, int, float, np.integer, np.floating)):
        return float(v)
    if isinstance(v, str):
        t = v.strip().lower().replace(",", ".")           
        return {"true": 1.0, "false": 0.0}.get(t) if t in ("true", "false") else float(t)
    raise TypeError("number nahi hai")


def validate_input(record, rules):
    """record = dict ya 13 values ki list. Return: {ok, x, errors, warnings, filled}"""
    out = {"ok": False, "x": None, "errors": [], "warnings": [], "filled": []}
    if isinstance(record, (list, tuple, np.ndarray)):
        if len(record) != len(FEATURES):
            out["errors"].append(f"{len(FEATURES)} values chahiye, {len(record)} mili")
            return out
        record = dict(zip(FEATURES, record))
    if not isinstance(record, dict):
        out["errors"].append("input dict ya list hona chahiye")
        return out

    rec = {str(k).strip().lower().replace(" ", "_"): v for k, v in record.items()}  
    extra = [k for k in rec if k not in rules]
    if extra:
        out["warnings"].append(f"extra fields ignore kiye: {extra}")

    x = np.zeros(len(FEATURES))
    for i, f in enumerate(FEATURES):
        v, r = rec.get(f), rules[f]
        if is_missing(v):
            x[i] = r["fill"]; out["filled"].append(f); continue
        try:
            val = to_number(v)
        except (ValueError, TypeError):
            out["errors"].append(f"{f}: '{v}' number nahi hai"); continue
        if not np.isfinite(val):
            out["errors"].append(f"{f}: inf/NaN allowed nahi"); continue
        if r["kind"] == "category":
            if val not in r["allowed"]:
                out["errors"].append(f"{f}: {val:g} allowed nahi (allowed: {[int(a) for a in r['allowed']]})")
        elif not r["hard_lo"] <= val <= r["hard_hi"]:
            out["errors"].append(f"{f}: {val:g} impossible value (limit {r['hard_lo']:.1f} se {r['hard_hi']:.1f})")
        elif not r["min"] <= val <= r["max"]:
            out["warnings"].append(f"{f}: {val:g} training range [{r['min']:g}, {r['max']:g}] ke bahar hai, prediction kam bharosemand")
        x[i] = val

    if len(out["filled"]) > MAX_MISSING:
        out["errors"].append(f"{len(out['filled'])} fields missing hain (max {MAX_MISSING} allowed)")
    elif out["filled"]:
        out["warnings"].append(f"missing fields median/mode se bhare gaye: {out['filled']}")
    if not out["errors"]:
        out["ok"], out["x"] = True, x
    return out


def predict_result(model, x):
    """x = 13 numbers. Pehle 2 pass (predict + predict_proba) the, ab sirf 1 pass."""
    probabilities = model.predict_proba(pd.DataFrame([x], columns=FEATURES))[0]
    prediction = int(model.classes_[np.argmax(probabilities)])
    return prediction, probabilities


def make_predictor(model, size=1024):
    """Same input dobara aaye to model dobara nahi chalta (cache)."""
    @lru_cache(maxsize=size)
    def cached(key):
        return predict_result(model, list(key))
    return lambda x: cached(tuple(float(v) for v in x))


def _to_numbers(s):
    """Series -> (numbers, missing_mask). Text wali value NaN banti hai aur missing nahi maani jati."""
    if pd.api.types.is_numeric_dtype(s):
        num = s.astype(float).to_numpy()
        return num, np.isnan(num)
    txt = s.astype(str).str.strip().str.lower()
    miss = (s.isna() | txt.isin(MISSING_WORDS)).to_numpy()
    num = pd.to_numeric(txt.str.replace(",", ".", regex=False).replace({"true": "1", "false": "0"}), errors="coerce").to_numpy(dtype=float)
    return num, miss


def validate_batch(raw, rules):
    """DataFrame ki saari rows ek saath validate. Return: X, ok(bool array), reason(Series)"""
    raw = raw.rename(columns=lambda c: str(c).strip().lower().replace(" ", "_"))
    n = len(raw)
    X, bad, n_miss = np.zeros((n, len(FEATURES))), np.zeros(n, bool), np.zeros(n, int)
    reason = pd.Series([""] * n, index=raw.index, dtype=object)
    for i, f in enumerate(FEATURES):
        r = rules[f]
        if f in raw:
            num, miss = _to_numbers(raw[f])
        else:
            num, miss = np.full(n, np.nan), np.ones(n, bool)
        bad_f = ~miss & ~np.isfinite(num)                               
        ok_num = ~miss & np.isfinite(num)
        if r["kind"] == "category":
            bad_f |= ok_num & ~np.isin(num, r["allowed"])
        else:
            bad_f |= ok_num & ((num < r["hard_lo"]) | (num > r["hard_hi"]))
        reason = reason + np.where(bad_f, f + " galat; ", "")
        bad |= bad_f; n_miss += miss
        X[:, i] = np.where(miss | bad_f, r["fill"], num)
    too_many = n_miss > MAX_MISSING
    reason = reason + np.where(too_many, "bahut fields missing; ", "")
    return X, ~(bad | too_many), reason


def predict_batch(model, rules, raw):
    """CSV/DataFrame ke liye: ek hi predict_proba call poori valid rows par."""
    X, ok, reason = validate_batch(raw, rules)
    out = pd.DataFrame({"status": np.where(ok, "ok", "rejected"), "reason": reason.str.strip("; ").to_numpy(),
                        "prediction": np.nan, "probability_%": np.nan}, index=raw.index)
    if ok.any():
        p = model.predict_proba(pd.DataFrame(X[ok], columns=FEATURES))
        out.loc[ok, "prediction"] = model.classes_[p.argmax(axis=1)]
        out.loc[ok, "probability_%"] = (p.max(axis=1) * 100).round(2)
    return out

Writing input.py


In [2]:
import warnings; warnings.filterwarnings('ignore')
import joblib
import pandas as pd
from input import *

model = joblib.load('heart_model_v20261013_160903.joblib')
rules = load_rules('heart.csv')

pd.DataFrame(rules).T[['kind', 'min', 'max', 'hard_lo', 'hard_hi', 'allowed', 'fill']]

,kind,min,max,hard_lo,hard_hi,allowed,fill
age,number,29.0,77.0,17.0,89.0,NaN,55.5
sex,category,0.0,1.0,NaN,NaN,"[0.0, 1.0]",1.0
cp,category,0.0,3.0,NaN,NaN,"[0.0, 1.0, 2.0, 3.0]",0.0
trestbps,number,94.0,200.0,67.5,226.5,NaN,130.0
chol,number,126.0,564.0,16.5,673.5,NaN,240.5
fbs,category,0.0,1.0,NaN,NaN,"[0.0, 1.0]",0.0
restecg,category,0.0,2.0,NaN,NaN,"[0.0, 1.0, 2.0]",1.0
thalach,number,71.0,202.0,38.25,234.75,NaN,152.5
exang,category,0.0,1.0,NaN,NaN,"[0.0, 1.0]",0.0
oldpeak,number,0.0,6.2,0.0,7.75,NaN,0.8


In [3]:
good = dict(pd.read_csv('heart.csv').iloc[0][FEATURES])       
def with_(**kw):
    d = dict(good); d.update(kw); return d

cases = {
    'valid':                (good, True),
    'numbers as text':      ({k: str(v) for k, v in good.items()}, True),
    'comma decimal 2,3':    (with_(oldpeak='2,3'), True),
    'keys alag case/space': ({' Age ': 63, 'SEX': 1, **{k: v for k, v in good.items() if k not in ('age', 'sex')}}, True),
    '13 values ki list':    (list(good.values()), True),
    '1 missing (None)':     (with_(chol=None), True),
    '3 missing (mixed)':    (with_(chol='', trestbps='NA', thal=float('nan')), True),
    '5 missing':            (with_(chol=None, trestbps=None, thal=None, ca=None, slope=None), False),
    'empty dict':           ({}, False),
    'list too short':       ([1, 2], False),
    'not dict/list':        ('abc', False),
    'text in number':       (with_(age='sixty'), False),
    'age = 250':            (with_(age=250), False),
    'negative chol':        (with_(chol=-5), False),
    'inf value':            (with_(thalach=float('inf')), False),
    'unknown category':     (with_(cp=9), False),
    'category 1.5':         (with_(cp=1.5), False),
    'chol 600 (range ke bahar)': (with_(chol=600), True),
    'extra field':          ({**good, 'name': 'Riya'}, True),
}
rows = []
for name, (rec, expected) in cases.items():
    r = validate_input(rec, rules)
    rows.append((name, expected, r['ok'], expected == r['ok'], (r['errors'] + r['warnings'] or [''])[0][:60]))
res = pd.DataFrame(rows, columns=['case', 'expected_ok', 'got_ok', 'match', 'message'])
print('Matched:', res['match'].sum(), 'of', len(res))
res

Matched: 19 of 19


,case,expected_ok,got_ok,match,message
0,valid,True,True,True,
1,numbers as text,True,True,True,
2,"comma decimal 2,3",True,True,True,
3,keys alag case/space,True,True,True,
4,13 values ki list,True,True,True,
5,1 missing (None),True,True,True,missing fields median/mode se bhare gaye: ['ch...
6,3 missing (mixed),True,True,True,missing fields median/mode se bhare gaye: ['tr...
7,5 missing,False,False,True,5 fields missing hain (max 3 allowed)
8,empty dict,False,False,True,13 fields missing hain (max 3 allowed)
9,list too short,False,False,True,"13 values chahiye, 2 mili"


In [4]:
messy = pd.DataFrame([good, with_(age='abc'), with_(chol=None), with_(cp=7), with_(trestbps=900),
                      with_(thal=None, ca=None, slope=None, oldpeak=None), good])
pd.concat([messy[['age', 'cp', 'chol', 'trestbps']], predict_batch(model, rules, messy)], axis=1)

,age,cp,chol,trestbps,status,reason,prediction,probability_%
0,63.0,3.0,233.0,145.0,ok,,1.0,78.0
1,abc,3.0,233.0,145.0,rejected,age galat,NaN,NaN
2,63.0,3.0,NaN,145.0,ok,,1.0,78.0
3,63.0,7.0,233.0,145.0,rejected,cp galat,NaN,NaN
4,63.0,3.0,233.0,900.0,rejected,trestbps galat,NaN,NaN
5,63.0,3.0,233.0,145.0,rejected,bahut fields missing,NaN,NaN
6,63.0,3.0,233.0,145.0,ok,,1.0,78.0


In [5]:
import time
records = pd.read_csv('heart.csv').drop_duplicates()[FEATURES].to_dict('records')[:100]

def old_way(rec):                                  
    d = pd.DataFrame([list(rec.values())], columns=FEATURES)
    return model.predict(d)[0], model.predict_proba(d)[0]

def new_way(rec):                                   
    return predict_result(model, validate_input(rec, rules)['x'])

predictor = make_predictor(model)
def new_cached(rec):                              
    return predictor(validate_input(rec, rules)['x'])

def ms_per_call(fn):
    t = time.perf_counter(); [fn(r) for r in records]; return (time.perf_counter() - t) / len(records) * 1000

ms_per_call(old_way)                               
speed = {'purana (2 pass, bina validation)': ms_per_call(old_way), 'naya: validation + 1 pass': ms_per_call(new_way)}
ms_per_call(new_cached); speed['naya + cache (same input dobara)'] = ms_per_call(new_cached)

big = pd.concat([pd.read_csv('heart.csv')[FEATURES]] * 100)       
t = time.perf_counter(); out = predict_batch(model, rules, big); speed['batch (30,300 rows, per row)'] = (time.perf_counter() - t) / len(big) * 1000

same = all(old_way(r)[0] == new_way(r)[0] for r in records)
print('Prediction purane tareeke jaisi hi hai:', same)
pd.Series(speed, name='ms per prediction').round(3)

Prediction purane tareeke jaisi hi hai: True


purana (2 pass, bina validation)    33.754
naya: validation + 1 pass           15.019
naya + cache (same input dobara)     0.102
batch (30,300 rows, per row)         0.015
Name: ms per prediction, dtype: float64

## Task 2: Display model predictions dynamically by implementing custom caching mechanisms to improve performance, ensuring smooth updates without unnecessary reloading, and handling concurrent user requests efficiently.

### Step 1: `cache.py`

In [6]:
%%writefile cache.py
"""
HeartCare - custom caching for predictions (Streamlit ka built-in cache nahi).

- LRU     : size limit; sabse purani entry hatti hai
- TTL     : entry kuch der baad expire (model/data purana na ho)
- Thread-safe : Lock se
- Single-flight : same input par 10 users ek saath aaye to model sirf 1 baar chalta hai,
                  baaki 9 wahi result ka wait karte hain (concurrent requests efficient)
"""
import time
import threading
from collections import OrderedDict


class _Call:
    """Jo computation chal rahi hai, uska result yahan rakhte hain."""
    def __init__(self):
        self.done = threading.Event()
        self.value = None
        self.error = None


class PredictionCache:

    def __init__(self, max_size=512, ttl=600):
        self.max_size = max_size
        self.ttl = ttl                      
        self._data = OrderedDict()         
        self._inflight = {}               
        self._lock = threading.Lock()
        self.hits = self.misses = self.waits = self.evictions = self.expired = 0

    def get_or_compute(self, key, compute):
        """Cache me ho to wahi do, warna compute() chalao (same key par ek hi baar)."""
        with self._lock:
            item = self._data.get(key)
            if item is not None:
                value, created = item
                if self.ttl is None or time.monotonic() - created < self.ttl:
                    self._data.move_to_end(key)        
                    self.hits += 1
                    return value
                del self._data[key]                      
                self.expired += 1

            call = self._inflight.get(key)
            owner = call is None
            if owner:
                call = self._inflight[key] = _Call()
                self.misses += 1
            else:
                self.waits += 1

        if owner:
            try:
                call.value = compute()                   
            except BaseException as error:
                call.error = error                     
            with self._lock:
                if call.error is None:
                    self._data[key] = (call.value, time.monotonic())
                    self._data.move_to_end(key)
                    while len(self._data) > self.max_size:
                        self._data.popitem(last=False)
                        self.evictions += 1
                del self._inflight[key]
            call.done.set()
        else:
            call.done.wait()

        if call.error is not None:
            raise call.error
        return call.value

    def clear(self):
        with self._lock:
            self._data.clear()

    def stats(self):
        with self._lock:
            total = self.hits + self.misses + self.waits
            return {"size": len(self._data), "hits": self.hits, "misses": self.misses,
                    "waits": self.waits, "evictions": self.evictions, "expired": self.expired,
                    "hit_rate_%": round(100 * (self.hits + self.waits) / total, 1) if total else 0.0}

Writing cache.py


In [7]:
import time
from cache import PredictionCache

small = PredictionCache(max_size=2, ttl=0.5)
runs = []
def work(v):
    runs.append(v); return v * 10

small.get_or_compute('a', lambda: work(1)); small.get_or_compute('a', lambda: work(1))    
small.get_or_compute('b', lambda: work(2)); small.get_or_compute('c', lambda: work(3))    
print('model/compute kitni baar chala:', len(runs), '(3 alag keys, 1 repeat)')
small.get_or_compute('a', lambda: work(1))                                                  
print('evicted key dobara compute hui:', runs.count(1) == 2)
time.sleep(0.6)
small.get_or_compute('c', lambda: work(3))                                                 
print(small.stats())

model/compute kitni baar chala: 3 (3 alag keys, 1 repeat)
evicted key dobara compute hui: True
{'size': 2, 'hits': 1, 'misses': 5, 'waits': 0, 'evictions': 2, 'expired': 1, 'hit_rate_%': 16.7}


In [8]:
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd

patients = pd.read_csv('heart.csv').drop_duplicates()[FEATURES].values[:10]
requests = [patients[i % 10] for i in range(60)]

def predict_p1(x):
    return model.predict_proba(pd.DataFrame([x], columns=FEATURES))[0, 1]

t = time.perf_counter()
with ThreadPoolExecutor(12) as pool: plain = list(pool.map(predict_p1, requests))
t_plain = time.perf_counter() - t

cache = PredictionCache(); runs = {'n': 0}
def cached_predict(x):
    def compute():
        runs['n'] += 1
        return predict_p1(x)
    return cache.get_or_compute(tuple(x), compute)

t = time.perf_counter()
with ThreadPoolExecutor(12) as pool: cached = list(pool.map(cached_predict, requests))
t_cache = time.perf_counter() - t

print(f'Bina cache : 60 model runs, {t_plain * 1000:.0f} ms')
print(f'Custom cache: {runs["n"]} model runs, {t_cache * 1000:.0f} ms')
print('Dono ke results same:', np.allclose(plain, cached))
print(cache.stats())

Bina cache : 60 model runs, 1498 ms
Custom cache: 10 model runs, 234 ms
Dono ke results same: True
{'size': 10, 'hits': 50, 'misses': 10, 'waits': 0, 'evictions': 0, 'expired': 0, 'hit_rate_%': 83.3}


## Task 3: Visualize prediction confidence using bar charts by manually computing probability distributions, implementing real-time updates with animations, and allowing users to interactively explore model confidence levels.

### Step 1: `confidence.py`

In [9]:
%%writefile confidence.py
"""
HeartCare - prediction confidence ko bar charts me dikhana.

1. Probability distribution HAATH se: Random Forest ki har tree ka probability nikal ke khud average karte hain
   (model.predict_proba ka use sirf check karne ke liye).
2. Charts HTML + CSS se bante hain. Har bar purani value se nayi value tak animate hota hai (CSS keyframes).
3. Interactive: threshold slider, what-if (ek feature badalke confidence kaise badalti hai).
"""
import numpy as np
import pandas as pd

FEATURES = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
            "thalach", "exang", "oldpeak", "slope", "ca", "thal"]
CLASS_NAMES = {0: "Lower Risk", 1: "Higher Risk"}      
COLORS = {0: "#2e9e6b", 1: "#d64545"}
N_BINS = 10


def analyse(model, x):
    """Ek patient ke liye poori confidence picture. Cache me yahi dict jati hai."""
    x32 = np.ascontiguousarray(x, dtype=np.float32).reshape(1, -1)             
    per_tree = np.array([t.predict_proba(x32, check_input=False)[0] for t in model.estimators_])
    probs = per_tree.mean(axis=0)                                               
    votes = np.bincount(per_tree.argmax(axis=1), minlength=len(probs))            
    hist = np.histogram(per_tree[:, 1], bins=N_BINS, range=(0, 1))[0]             
    return {"probs": probs, "votes": votes, "hist": hist, "n_trees": len(per_tree),
            "std": float(per_tree[:, 1].std()), "x": np.asarray(x, dtype=float)}


def decide(probs, threshold=0.5):
    """Threshold ke hisaab se class (default 0.5 = model.predict jaisa)."""
    return int(probs[1] > threshold)


def sweep(model, x, feature, values):
    """What-if: ek feature ki value badalke P(class 1) (ek hi predict_proba call me)."""
    rows = np.tile(np.asarray(x, dtype=float), (len(values), 1))
    rows[:, FEATURES.index(feature)] = values
    return model.predict_proba(pd.DataFrame(rows, columns=FEATURES))[:, 1]


def sweep_values(rule, n=10):
    """Rules (input.py) se feature ki values: category = allowed, number = n barabar points."""
    if rule["kind"] == "category":
        return [float(v) for v in rule["allowed"]]
    return [float(v) for v in np.round(np.linspace(rule["min"], rule["max"], n), 1)]


CSS = """
<style>
.hc-chart { background: white; border: 1px solid #e2e8f0; border-radius: 12px; padding: 16px; margin-bottom: 14px; }
.hc-chart h4 { margin: 0 0 12px 0; color: #142b53; font-size: 16px; }
.hc-row { display: flex; align-items: center; margin: 8px 0; }
.hc-label { width: 120px; font-size: 14px; color: #334155; }
.hc-track { position: relative; flex: 1; height: 22px; background: #eef2f7; border-radius: 6px; overflow: hidden; }
.hc-fill { height: 100%; width: var(--to); animation: hc-grow 0.9s ease-out; }
.hc-value { width: 64px; text-align: right; font-size: 14px; font-weight: 600; color: #142b53; }
.hc-thr { position: absolute; top: 0; bottom: 0; width: 2px; background: #142b53; }
.hc-cols { display: flex; align-items: flex-end; height: 130px; gap: 6px; border-bottom: 1px solid #cbd5e1; }
.hc-colwrap { flex: 1; height: 100%; display: flex; flex-direction: column; justify-content: flex-end; align-items: center; }
.hc-col { width: 100%; height: var(--to); border-radius: 4px 4px 0 0; animation: hc-rise 0.9s ease-out; }
.hc-colnum { font-size: 11px; color: #475569; height: 14px; }
.hc-xlabels { display: flex; gap: 6px; margin-top: 4px; }
.hc-xlabels span { flex: 1; text-align: center; font-size: 11px; color: #475569; }
.hc-note { font-size: 12px; color: #64748b; margin-top: 8px; }
@keyframes hc-grow { from { width: var(--from); } to { width: var(--to); } }
@keyframes hc-rise { from { height: var(--from); } to { height: var(--to); } }
@media (max-width: 600px) { .hc-label { width: 84px; font-size: 12px; } .hc-value { width: 52px; } }
@media (prefers-reduced-motion: reduce) { .hc-fill, .hc-col { animation: none; } }
</style>
"""


def _prev(prev, i, scale):
    """Purani value (animation ka start). Pehli baar 0 se grow hota hai."""
    return 0.0 if prev is None or i >= len(prev) else float(prev[i]) * scale


def bar_chart(title, labels, values, prev=None, colors=None, threshold=None, note=""):
    """Horizontal bars. values 0-1 (probability). threshold = vertical line."""
    rows = ""
    for i, (lab, v) in enumerate(zip(labels, values)):
        color = colors[i] if colors else "#244577"
        thr = f'<div class="hc-thr" style="left:{threshold * 100:.1f}%"></div>' if threshold is not None else ""
        rows += (f'<div class="hc-row"><div class="hc-label">{lab}</div>'
                 f'<div class="hc-track"><div class="hc-fill" style="--from:{_prev(prev, i, 100):.1f}%;--to:{v * 100:.1f}%;background:{color}"></div>{thr}</div>'
                 f'<div class="hc-value">{v * 100:.1f}%</div></div>')
    note_html = f'<div class="hc-note">{note}</div>' if note else ""
    return f'<div class="hc-chart"><h4>{title}</h4>{rows}{note_html}</div>'


def column_chart(title, x_labels, values, prev=None, color="#244577", counts=None, note=""):
    """Vertical bars. values 0-1 (height ka fraction); counts = bar ke upar likha number."""
    cols, labs = "", ""
    for i, (lab, v) in enumerate(zip(x_labels, values)):
        shown = counts[i] if counts is not None else f"{v * 100:.0f}%"
        cols += (f'<div class="hc-colwrap"><div class="hc-colnum">{shown}</div>'
                 f'<div class="hc-col" style="--from:{_prev(prev, i, 100):.1f}%;--to:{v * 100:.1f}%;background:{color}"></div></div>')
        labs += f"<span>{lab}</span>"
    note_html = f'<div class="hc-note">{note}</div>' if note else ""
    return f'<div class="hc-chart"><h4>{title}</h4><div class="hc-cols">{cols}</div><div class="hc-xlabels">{labs}</div>{note_html}</div>'

Writing confidence.py


In [10]:
import numpy as np
import importlib, confidence
importlib.reload(confidence)
from confidence import *

X_all = pd.read_csv('heart.csv').drop_duplicates()[FEATURES].values
manual = np.array([analyse(model, x)['probs'] for x in X_all])
sk = model.predict_proba(pd.DataFrame(X_all, columns=FEATURES))
print('Patients:', len(X_all), '| max |manual - sklearn| =', np.abs(manual - sk).max())
print('Prediction same (threshold 50%):', all(decide(p) == model.classes_[k] for p, k in zip(manual, sk.argmax(1))))

Patients: 302 | max |manual - sklearn| = 0.0
Prediction same (threshold 50%): True


In [11]:
from IPython.display import HTML

x = X_all[0]
info = analyse(model, x)
labels = [CLASS_NAMES[0], CLASS_NAMES[1]]
print('Probabilities:', info['probs'], '| votes:', info['votes'], '| tree spread (std):', round(info['std'], 3))

HTML(CSS
     + bar_chart('Class probabilities (trees ka average)', labels, info['probs'], colors=[COLORS[0], COLORS[1]], threshold=0.5)
     + bar_chart('Tree votes', labels, info['votes'] / info['n_trees'], colors=[COLORS[0], COLORS[1]])
     + column_chart('Har tree ka Higher Risk probability (%)', [f'{i*10}-{i*10+10}' for i in range(10)],
                    info['hist'] / info['hist'].max(), counts=[int(h) for h in info['hist']]))

Probabilities: [0.22 0.78] | votes: [22 78] | tree spread (std): 0.414


In [12]:
for thr in (0.3, 0.5, 0.7, 0.9):
    print(f'threshold {thr:.0%}: prediction = {CLASS_NAMES[decide(info["probs"], thr)]}')

rules = load_rules('heart.csv')
feature = 'cp'
values = sweep_values(rules[feature])
p_values = sweep(model, x, feature, values)
HTML(CSS + column_chart(f'What-if: {feature} badalne par {CLASS_NAMES[1]} probability', [f'{v:g}' for v in values], p_values, color='#d64545',
                        note='Baaki saari values wahi rakhi gayi hain; sirf ' + feature + ' badla.'))

threshold 30%: prediction = Higher Risk
threshold 50%: prediction = Higher Risk
threshold 70%: prediction = Higher Risk
threshold 90%: prediction = Lower Risk


In [13]:
%%writefile app_v1.py

import time
import joblib
import numpy as np
import pandas as pd
import streamlit as st

from input import FEATURES, load_rules, validate_input, predict_batch     
from cache import PredictionCache                                        
from confidence import (CSS, CLASS_NAMES, COLORS, analyse, decide, sweep, 
                        sweep_values, bar_chart, column_chart)

st.set_page_config(
    page_title="HeartCare",
    layout="centered"
)

MODEL_FILE = "heart_model_v20261013_160903.joblib"
DATA_FILE = "heart.csv"
MAX_BATCH_ROWS = 100000


@st.cache_resource
def load_everything():
    model = joblib.load(MODEL_FILE)
    rules = load_rules(DATA_FILE)
    cache = PredictionCache(max_size=512, ttl=600)
    return model, rules, cache, MODEL_FILE


try:
    model, rules, cache, model_version = load_everything()
except FileNotFoundError as e:
    st.error(f"File nahi mili: {e.filename}. app.py ke same folder me rakho.")
    st.stop()


def hint(f):
    r = rules[f]
    if r["kind"] == "category":
        return "/".join(str(int(a)) for a in r["allowed"])
    return f"{r['min']:g} to {r['max']:g}"


def choice(label, f):
    return st.selectbox(f"{label} ({hint(f)})", [None] + rules[f]["allowed"],
                        index=1, format_func=lambda v: "Not known" if v is None else str(int(v)))


def number(label, f, default):
    return st.text_input(f"{label} ({hint(f)})", value=default)


def previous(key, current):
    """Chart ki pichli values (animation start). Session me yaad rakhte hain."""
    old = st.session_state.get("prev", {}).get(key)
    st.session_state.setdefault("prev", {})[key] = list(current)
    return old


st.markdown("""
<style>
.stApp {
    background: #f3f6fb;
}

header, footer {
    visibility: hidden;
}

.main .block-container {
    max-width: 850px;
    padding: 25px 20px;
}

.header {
    background: #142b53;
    color: white;
    padding: 25px;
    border-radius: 14px;
    text-align: center;
    margin-bottom: 25px;
}

.section {
    background: white;
    padding: 20px;
    border-radius: 12px;
    margin-bottom: 18px;
    border: 1px solid #e2e8f0;
}

.section h3 {
    color: #142b53;
}

.stButton button {
    background: #142b53;
    color: white;
    width: 100%;
    height: 45px;
    border-radius: 8px;
}

.stButton button:hover {
    background: #244577;
    color: white;
}

@media(max-width: 600px) {
    .main .block-container {
        padding: 12px 10px;
    }

    .header {
        padding: 18px;
    }

    .header h1 {
        font-size: 24px;
    }

    .section {
        padding: 14px;
    }
}
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="header">
    <h1>HeartCare</h1>
    <p>Heart Disease Risk Prediction System</p>
</div>
""", unsafe_allow_html=True)

st.markdown("""
<div class="section">
    <h3>Patient Information</h3>
    <p>Enter patient medical details below.</p>
</div>
""", unsafe_allow_html=True)

st.caption("Koi field khali chhodo ya 'Not known' chuno to median/mode se bhar diya jayega (max 3 fields).")

age = number("Age", "age", "45")
sex = choice("Sex", "sex")
cp = choice("Chest Pain Type", "cp")
trestbps = number("Resting Blood Pressure", "trestbps", "120")
chol = number("Cholesterol", "chol", "200")
fbs = choice("Fasting Blood Sugar", "fbs")
restecg = choice("Resting ECG", "restecg")
thalach = number("Maximum Heart Rate", "thalach", "150")
exang = choice("Exercise Induced Angina", "exang")
oldpeak = number("Oldpeak", "oldpeak", "1.0")
slope = choice("Slope", "slope")
ca = choice("Major Vessels", "ca")
thal = choice("Thal", "thal")

live = st.checkbox("Live update (input badalte hi prediction update ho)")
clicked = st.button("Predict Heart Disease Risk")

if clicked or live:

    record = dict(zip(FEATURES, [
        age, sex, cp, trestbps, chol,
        fbs, restecg, thalach, exang,
        oldpeak, slope, ca, thal
    ]))

    check = validate_input(record, rules)

    for message in check["errors"]:
        st.error(message)

    if check["ok"]:
        st.session_state["check"] = check      
    else:
        st.session_state.pop("check", None)

check = st.session_state.get("check")

if check:

    st.markdown(CSS, unsafe_allow_html=True)

    st.markdown("""
<div class="section">
    <h3>Prediction & Confidence</h3>
</div>
""", unsafe_allow_html=True)

    threshold = st.slider("Decision threshold (%): Higher Risk tab maana jaye jab probability isse zyada ho", 1, 99, 50) / 100
    explore = st.selectbox("What-if: ek feature badalke confidence dekho", ["(none)"] + FEATURES)

    start = time.perf_counter()
    key = (model_version, tuple(float(v) for v in check["x"]))
    info = cache.get_or_compute(key, lambda: analyse(model, check["x"]))
    elapsed = (time.perf_counter() - start) * 1000

    for message in check["warnings"]:
        st.warning(message)

    probs = info["probs"]
    prediction = decide(probs, threshold)

    if prediction == 1:
        st.error(f"{CLASS_NAMES[1]} Prediction")
    else:
        st.success(f"{CLASS_NAMES[0]} Prediction")

    st.write("Model Confidence:", round(max(probs) * 100, 2), "%")

    if threshold != 0.5:
        st.caption(f"Custom threshold {threshold * 100:.0f}%: {CLASS_NAMES[1]} probability {probs[1] * 100:.1f}% hai, "
                   f"isliye prediction {CLASS_NAMES[prediction]} dikh raha hai (model ka default threshold 50% hai).")
    st.caption(f"{info['votes'][1]} of {info['n_trees']} trees ne {CLASS_NAMES[1]} chuna | trees ke beech spread (std): {info['std']:.2f}")

    labels = [CLASS_NAMES[0], CLASS_NAMES[1]]
    st.markdown(bar_chart("Class probabilities (trees ka average)", labels, probs,
                          prev=previous("probs", probs), colors=[COLORS[0], COLORS[1]], threshold=threshold,
                          note=f"Kali line = threshold ({threshold * 100:.0f}%)"), unsafe_allow_html=True)

    votes = info["votes"] / info["n_trees"]
    st.markdown(bar_chart("Tree votes (kitni trees ne kaunsi class chuni)", labels, votes,
                          prev=previous("votes", votes), colors=[COLORS[0], COLORS[1]]), unsafe_allow_html=True)

    hist = info["hist"]
    bins = [f"{i * 10}-{i * 10 + 10}" for i in range(len(hist))]
    st.markdown(column_chart(f"Har tree ka {CLASS_NAMES[1]} probability (%) - kitni trees kis range me",
                             bins, hist / hist.max(), prev=previous("hist", hist / hist.max()),
                             color="#244577", counts=[int(h) for h in hist],
                             note="Bar ke upar tree ginti. Sab trees ek jaisi ho to bars sirf kinaron par aate hain."),
                unsafe_allow_html=True)

    if explore != "(none)":
        values = sweep_values(rules[explore])
        p_values = sweep(model, check["x"], explore, values)
        st.markdown(column_chart(f"What-if: {explore} badalne par {CLASS_NAMES[1]} probability",
                                 [f"{v:g}" for v in values], p_values, prev=previous("sweep_" + explore, p_values),
                                 color="#d64545", note=f"Baaki saari values wahi rakhi gayi hain; sirf {explore} badla."),
                    unsafe_allow_html=True)

    if not live:
        st.caption("Ye result pichhle Predict click ke inputs ka hai. Naye inputs ke liye dobara Predict dabao ya Live update on karo.")

    stats = cache.stats()
    st.caption(f"Is request me {elapsed:.2f} ms | Cache: {stats['hits']} hits, {stats['misses']} misses, "
               f"{stats['waits']} coalesced, hit rate {stats['hit_rate_%']}% (size {stats['size']})")

st.markdown("""
<div class="section">
    <h3>Batch Prediction (CSV)</h3>
    <p>Bahut patients ek saath: CSV me columns age, sex, cp, trestbps, chol, fbs,
    restecg, thalach, exang, oldpeak, slope, ca, thal hone chahiye.</p>
</div>
""", unsafe_allow_html=True)

uploaded = st.file_uploader("Upload CSV", type=["csv"])

data = None

if uploaded is not None:

    try:
        data = pd.read_csv(uploaded)
    except Exception:
        st.error("CSV padh nahi paye. File sahi CSV format me honi chahiye.")

if data is not None:

    clean_names = [str(c).strip().lower().replace(" ", "_") for c in data.columns]
    missing_cols = [f for f in FEATURES if f not in clean_names]

    if len(data) == 0:
        st.error("CSV khali hai.")
    elif len(data) > MAX_BATCH_ROWS:
        st.error(f"Max {MAX_BATCH_ROWS} rows allowed hain.")
    else:
        if missing_cols:
            st.warning(f"Ye columns CSV me nahi mile (missing maane gaye): {missing_cols}")

        start = time.perf_counter()
        results = predict_batch(model, rules, data)
        elapsed = (time.perf_counter() - start) * 1000

        rejected = int((results["status"] == "rejected").sum())
        st.write(f"Rows: {len(data)} | Predicted: {len(data) - rejected} | Rejected: {rejected}")
        st.caption(f"{elapsed:.0f} ms ({elapsed / len(data):.3f} ms per row)")

        output = pd.concat([data, results], axis=1)
        st.dataframe(output.head(100))

        st.download_button("Download results", output.to_csv(index=False), "heart_predictions.csv", "text/csv")

st.markdown("""
<div style="text-align:center;color:#64748b;padding:20px;">
    HeartCare | Educational Purpose Only
    <br>
    Not a substitute for professional medical diagnosis.
</div>
""", unsafe_allow_html=True)

Writing app_v1.py


## Task 4: Test UI for usability by conducting user experience research, gathering qualitative feedback, identifying pain points in interaction flow, and iteratively refining the interface based on usability heuristics.

In [14]:
import pandas as pd
v1_issues = pd.DataFrame([
    ('P1', 'Age/BP/Chol/Thalach/Oldpeak text box me type karne padte hain; galat type karo to Predict ke baad hi error milta hai', 'H5 Error prevention', 3, 'Number fields ab slider (range data se)'),
    ('P2', 'Predict dabane ke baad result 13 fields ke NEECHE aata hai; scroll karna padta hai', 'H1 Visibility of system status', 3, 'Result card form ke UPAR (st.empty) - slider hilate hi dikhta hai'),
    ('P3', 'Live update default OFF; result purane input ka hota hai (stale)', 'H1 Visibility of system status', 2, 'Live update default ON + risk card par pichhle update se farak'),
    ('P4', 'Sab inputs ek saath dobara default par laane ka koi button nahi', 'H3 User control and freedom', 2, 'Reset to typical values button'),
    ('P5', '0/1/2/3 codes bina hint ke (sex, cp, thal ...)', 'H2 Match with real world / H10 Help', 3, 'Har field par hover/tap help (?). Poora meaning dataset me documented nahi - open issue'),
    ('P6', '"Not known" kabhi selectbox me, kabhi khali text box - alag-alag tareeke', 'H4 Consistency', 2, 'Ek jagah: Values I don\'t know (max 3) + estimated marker'),
    ('P7', '13 fields ek lambi flat list', 'H8 Aesthetic and minimalist design', 2, '4 chhote groups (Basic / Chest / Tests / Scan)'),
    ('P8', 'Prediction kyun aaya - koi explanation nahi', 'H1 / H10', 2, 'SHAP section: har feature ka contribution'),
    ('P9', 'Value kitni "normal" hai ye pata nahi chalta', 'H6 Recognition rather than recall', 1, 'Har slider ke neeche percentile line'),
    ('P10', 'App par feedback dene ki jagah nahi', 'H9 / research', 1, 'Feedback form -> feedback.csv'),
], columns=['id', 'pain point', 'heuristic', 'severity', 'fix in v2']).sort_values('severity', ascending=False)
print('Total issues:', len(v1_issues), '| severity >= 3:', (v1_issues.severity >= 3).sum())
v1_issues

Total issues: 10 | severity >= 3: 3


,id,pain point,heuristic,severity,fix in v2
0,P1,Age/BP/Chol/Thalach/Oldpeak text box me type k...,H5 Error prevention,3,Number fields ab slider (range data se)
1,P2,Predict dabane ke baad result 13 fields ke NEE...,H1 Visibility of system status,3,Result card form ke UPAR (st.empty) - slider h...
4,P5,"0/1/2/3 codes bina hint ke (sex, cp, thal ...)",H2 Match with real world / H10 Help,3,Har field par hover/tap help (?). Poora meanin...
2,P3,Live update default OFF; result purane input k...,H1 Visibility of system status,2,Live update default ON + risk card par pichhle...
3,P4,Sab inputs ek saath dobara default par laane k...,H3 User control and freedom,2,Reset to typical values button
5,P6,"""Not known"" kabhi selectbox me, kabhi khali te...",H4 Consistency,2,Ek jagah: Values I don't know (max 3) + estima...
6,P7,13 fields ek lambi flat list,H8 Aesthetic and minimalist design,2,4 chhote groups (Basic / Chest / Tests / Scan)
7,P8,Prediction kyun aaya - koi explanation nahi,H1 / H10,2,SHAP section: har feature ka contribution
8,P9,"Value kitni ""normal"" hai ye pata nahi chalta",H6 Recognition rather than recall,1,Har slider ke neeche percentile line
9,P10,App par feedback dene ki jagah nahi,H9 / research,1,Feedback form -> feedback.csv


In [15]:
v2_open = pd.DataFrame([
    ('Q1', 'Categories ke codes (sex, cp, thal...) ka asli meaning dataset me nahi diya; sirf "code 0-3" bata paate hain', 'H2 Match with real world', 3, 'Dataset documentation chahiye; tab tak label nahi badal sakte (galat label ka risk)'),
    ('Q2', 'Phone par Cholesterol slider (126-564) me ek-ek unit ka fine control mushkil ho sakta hai', 'H7 Flexibility and efficiency', 2, 'Real users ke saath phone par test chahiye; zaroorat ho to number box add karo'),
    ('Q3', '"pts" (percentage points) shabd naye user ko samajh na aaye', 'H2 Match with real world', 2, 'Feedback form me poocho; "points" ka example line add kar sakte hain'),
    ('Q4', 'Slider sirf dataset range (min-max) tak jaata hai; CSV batch wider range accept karta hai', 'H4 Consistency', 1, 'Intentional: single patient UI ko safe rakha'),
    ('Q5', 'Koi real user test nahi hua - upar ke saare ratings mera apna evaluation hain', 'Method limit', 4, 'Feedback form + study plan (neeche) se real users se data lo'),
], columns=['id', 'open issue', 'heuristic', 'severity', 'next step'])
print('v1 issues fixed (P1-P4, P6-P10): 9 of 10 | P5 partly fixed (tooltip) -> Q1')
v2_open

v1 issues fixed (P1-P4, P6-P10): 9 of 10 | P5 partly fixed (tooltip) -> Q1


,id,open issue,heuristic,severity,next step
0,Q1,"Categories ke codes (sex, cp, thal...) ka asli...",H2 Match with real world,3,Dataset documentation chahiye; tab tak label n...
1,Q2,Phone par Cholesterol slider (126-564) me ek-e...,H7 Flexibility and efficiency,2,Real users ke saath phone par test chahiye; za...
2,Q3,"""pts"" (percentage points) shabd naye user ko s...",H2 Match with real world,2,"Feedback form me poocho; ""points"" ka example l..."
3,Q4,Slider sirf dataset range (min-max) tak jaata ...,H4 Consistency,1,Intentional: single patient UI ko safe rakha
4,Q5,Koi real user test nahi hua - upar ke saare ra...,Method limit,4,Feedback form + study plan (neeche) se real us...


In [16]:
%%writefile feedback.py
"""
HeartCare - usability feedback ikatha karna aur analyze karna.

- add_feedback : app ke andar se feedback CSV me (thread-safe, health inputs store nahi hote)
- summarize    : success rate, ease rating, device-wise, aur comments ke themes (manual keyword coding)
- sus_score    : System Usability Scale (10 sawaal) ka standard scoring
"""
import csv
import os
import threading
import time
from collections import Counter

import pandas as pd

FIELDS = ["time", "version", "device", "task_success", "ease_1to5", "comment"]
_LOCK = threading.Lock()

THEMES = {
    "labels/jargon":  ["confus", "unclear", "jargon", "term", "medical", "understand", "samajh", "meaning", "label"],
    "length/scroll":  ["long", "scroll", "many", "too much", "lamba", "lambi", "zyada"],
    "mobile/touch":   ["phone", "mobile", "small", "touch", "tap"],
    "speed":          ["slow", "lag", "wait", "loading", "dhima"],
    "charts/results": ["chart", "graph", "bar", "shap", "result", "explanation"],
    "inputs":         ["slider", "type", "typing", "input", "field", "default"],
}


def _safe(text, limit=500):
    """CSV injection se bachao: =,+,-,@ se shuru text ke aage ' laga do. Length limit."""
    text = str(text).replace("\r", " ").replace("\n", " ").strip()[:limit]
    return "'" + text if text[:1] in ("=", "+", "-", "@") else text


def add_feedback(path, device, task_success, ease, comment="", version="v2"):
    """Ek feedback row save karo. Galat values par ValueError."""
    if task_success not in ("Yes", "Partly", "No"):
        raise ValueError("task_success: Yes / Partly / No")
    if not 1 <= int(ease) <= 5:
        raise ValueError("ease 1 se 5 ke beech hona chahiye")
    row = [time.strftime("%Y-%m-%d %H:%M:%S"), version, _safe(device, 30), task_success, int(ease), _safe(comment)]
    with _LOCK:                                                      
        new = not os.path.exists(path)
        with open(path, "a", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            if new:
                w.writerow(FIELDS)
            w.writerow(row)


def load_feedback(path):
    return pd.read_csv(path) if os.path.exists(path) else pd.DataFrame(columns=FIELDS)


def summarize(df):
    """Feedback ka summary. Kam responses (n<5) par conclusions pakke nahi hote, isliye n bhi return hota hai."""
    if len(df) == 0:
        return {"n": 0}
    themes = Counter()
    for c in df["comment"].fillna("").str.lower():
        for name, words in THEMES.items():
            if any(w in c for w in words):
                themes[name] += 1
    return {"n": len(df),
            "success_rate_%": round((df["task_success"] == "Yes").mean() * 100, 1),
            "mean_ease": round(df["ease_1to5"].mean(), 2),
            "by_device": df.groupby("device")["ease_1to5"].agg(["count", "mean"]).round(2).to_dict("index"),
            "themes": dict(themes.most_common())}


def sus_score(answers):
    """SUS: 10 jawab (1-5). Odd sawaal: jawab-1, even sawaal: 5-jawab, total x 2.5 -> 0-100."""
    if len(answers) != 10 or any(not 1 <= a <= 5 for a in answers):
        raise ValueError("SUS ke liye 10 jawab chahiye, har ek 1 se 5")
    return sum((a - 1) if i % 2 == 0 else (5 - a) for i, a in enumerate(answers)) * 2.5

Overwriting feedback.py


In [17]:
import os, tempfile, importlib
import feedback; importlib.reload(feedback)
from feedback import add_feedback, load_feedback, summarize, sus_score

tmp = os.path.join(tempfile.mkdtemp(), 'test_feedback.csv')
tests = [('Phone', 'Yes', 5, 'bahut aasaan'), ('Phone', 'Partly', 2, 'slider chhota hai, value set karna mushkil'),
         ('Laptop', 'Yes', 4, ''), ('Tablet', 'No', 1, 'result samajh nahi aaya, confusing'),
         ('Phone', 'Yes', 3, '=HYPERLINK("http://x")  <- CSV injection test')]
for d, s, e, c in tests: add_feedback(tmp, d, s, e, c)
print('--- bad input reject hota hai?')
for args in [('Phone', 'Maybe', 3), ('Phone', 'Yes', 9)]:
    try: add_feedback(tmp, *args)
    except ValueError as err: print('  rejected:', args, '->', err)
print('--- summary (sirf TEST data)')
df_fb = load_feedback(tmp); s = summarize(df_fb); print(s)
print('--- injection saaf hua?', df_fb.comment.iloc[-1])
print('--- SUS check: all best =', sus_score([5, 1] * 5), '| all neutral =', sus_score([3] * 10), '| all worst =', sus_score([1, 5] * 5))

--- bad input reject hota hai?
  rejected: ('Phone', 'Maybe', 3) -> task_success: Yes / Partly / No
  rejected: ('Phone', 'Yes', 9) -> ease 1 se 5 ke beech hona chahiye
--- summary (sirf TEST data)
{'n': 5, 'success_rate_%': np.float64(60.0), 'mean_ease': np.float64(3.0), 'by_device': {'Laptop': {'count': 1, 'mean': 4.0}, 'Phone': {'count': 3, 'mean': 3.33}, 'Tablet': {'count': 1, 'mean': 1.0}}, 'themes': {'inputs': 1, 'labels/jargon': 1, 'charts/results': 1}}
--- injection saaf hua? '=HYPERLINK("http://x")  <- CSV injection test
--- SUS check: all best = 100.0 | all neutral = 50.0 | all worst = 0.0


## Task 5: Add interactive graphs and sliders for inputs by implementing custom visualization components, ensuring compatibility with multiple device types, and dynamically linking input changes to real-time model updates.

In [18]:
%%writefile components.py
"""
HeartCare - custom visualization components (HTML + inline SVG + CSS, koi chart library nahi).

- risk_card   : live result card (sliders hilte hi update; purani value se nayi tak animate)
- curve_chart : interactive risk curve (SVG): population histogram + model ka risk curve + "aap yahan hain" marker
- percentile_text : slider ke neeche "aap dataset me kahan hain"

Multiple devices: SVG viewBox se chart apne aap scale hota hai, cards flex-wrap karte hain,
fonts mobile par chhote nahi hote, animations 'reduced motion' setting ko maante hain.
"""
import numpy as np

from confidence import CLASS_NAMES, COLORS

CSS = """
<style>
.ui-card { background: white; border: 1px solid #e2e8f0; border-radius: 14px; padding: 16px 18px; margin-bottom: 14px; }
.ui-top { display: flex; flex-wrap: wrap; align-items: center; gap: 12px 20px; }
.ui-big { font-size: 40px; font-weight: 700; line-height: 1.1; color: #142b53; }
.ui-label { font-size: 16px; font-weight: 600; }
.ui-sub { font-size: 13px; color: #64748b; }
.ui-meter { position: relative; height: 14px; border-radius: 7px; margin-top: 14px; background: linear-gradient(90deg, #2e9e6b, #f2c94c, #d64545); }
.ui-pin { position: absolute; top: -5px; width: 6px; height: 24px; border-radius: 3px; background: #142b53; left: var(--to); margin-left: -3px; animation: ui-move 0.8s ease-out; }
.ui-thr { position: absolute; top: -2px; bottom: -2px; width: 2px; background: #475569; opacity: .7; }
.ui-scale { display: flex; justify-content: space-between; font-size: 11px; color: #64748b; margin-top: 4px; }
.ui-svg { width: 100%; height: auto; max-width: 640px; display: block; margin: 0 auto; }
.ui-svg text { font-family: inherit; fill: #475569; }
.ui-line { fill: none; stroke: #d64545; stroke-width: 2.5; stroke-linejoin: round; stroke-dasharray: 1; stroke-dashoffset: 0; animation: ui-draw 1s ease-out; }
.ui-pt:hover { r: 6; }
@keyframes ui-move { from { left: var(--from); } to { left: var(--to); } }
@keyframes ui-draw { from { stroke-dashoffset: 1; } to { stroke-dashoffset: 0; } }
@media (max-width: 600px) { .ui-big { font-size: 32px; } .ui-card { padding: 12px; } }
@media (prefers-reduced-motion: reduce) { .ui-pin, .ui-line { animation: none; } }
</style>
"""


def risk_card(prob, threshold=0.5, prev_prob=None, estimated=()):
    """Top par live card: P(Higher Risk), decision, pichle update se farak, meter."""
    decision = 1 if prob > threshold else 0
    start = prob if prev_prob is None else prev_prob
    delta = "" if prev_prob is None or abs(prob - prev_prob) < 1e-9 else \
        f'<span class="ui-sub">{"&#9650;" if prob > prev_prob else "&#9660;"} {abs(prob - prev_prob) * 100:.1f} pts pichhle update se</span>'
    est = f'<div class="ui-sub">Estimated (aapne nahi diya): {", ".join(estimated)}</div>' if estimated else ""
    return (f'<div class="ui-card"><div class="ui-top"><div class="ui-big">{prob * 100:.1f}%</div>'
            f'<div><div class="ui-label" style="color:{COLORS[decision]}">{CLASS_NAMES[decision]}</div>'
            f'<div class="ui-sub">{CLASS_NAMES[1]} probability (threshold {threshold * 100:.0f}%)</div>{delta}</div></div>'
            f'<div class="ui-meter"><div class="ui-thr" style="left:{threshold * 100:.1f}%"></div>'
            f'<div class="ui-pin" style="--from:{start * 100:.1f}%;--to:{prob * 100:.1f}%"></div></div>'
            f'<div class="ui-scale"><span>0%</span><span>50%</span><span>100%</span></div>{est}</div>')


def curve_chart(title, feature, xs, ps, current_x, current_p, population, is_category, threshold=0.5):
    """SVG risk curve. xs/ps = feature ki values aur model ka P(Higher Risk); population = dataset ki us feature ki values."""
    W, H, L, R, T, B = 380, 240, 44, 10, 14, 46
    pw, ph = W - L - R, H - T - B
    xs = np.asarray(xs, float); ps = np.asarray(ps, float)
    lo, hi = xs.min(), xs.max()
    sx = (lambda v: L + (np.where(np.asarray(xs) == v)[0][0] + 0.5) / len(xs) * pw) if is_category else \
         (lambda v: L + (float(v) - lo) / (hi - lo + 1e-9) * pw)
    sy = lambda p: T + (1 - float(p)) * ph
    out = [f'<svg class="ui-svg" viewBox="0 0 {W} {H}" role="img" aria-label="{title}">']
    for g in (0, .25, .5, .75, 1):                                                  
        out.append(f'<line x1="{L}" x2="{W - R}" y1="{sy(g):.1f}" y2="{sy(g):.1f}" stroke="#e2e8f0"/><text x="{L - 6}" y="{sy(g) + 4:.1f}" font-size="13" text-anchor="end">{g * 100:.0f}%</text>')
    pop = np.asarray(population, float)                                           
    if is_category:
        counts = np.array([(pop == v).sum() for v in xs]); edges = None
    else:
        counts, edges = np.histogram(pop, bins=12, range=(lo, hi))
    top = counts.max() or 1
    for i, c in enumerate(counts):
        if is_category:
            x0, bw = L + i / len(xs) * pw + 4, pw / len(xs) - 8
        else:
            x0, bw = L + (edges[i] - lo) / (hi - lo + 1e-9) * pw + 1, (edges[i + 1] - edges[i]) / (hi - lo + 1e-9) * pw - 2
        bh = c / top * ph * 0.35
        out.append(f'<rect x="{x0:.1f}" y="{T + ph - bh:.1f}" width="{max(bw, 1):.1f}" height="{bh:.1f}" fill="#cbd5e1" opacity=".6"><title>{int(c)} patients</title></rect>')
    ty = sy(threshold)                                                             
    out.append(f'<line x1="{L}" x2="{W - R}" y1="{ty:.1f}" y2="{ty:.1f}" stroke="#475569" stroke-dasharray="4 4"/><text x="{W - R}" y="{ty - 4:.1f}" font-size="12" text-anchor="end">threshold</text>')
    pts = " ".join(f"{sx(x):.1f},{sy(p):.1f}" for x, p in zip(xs, ps))
    out.append(f'<polyline class="ui-line" pathLength="1" points="{pts}"/>')
    for x, p in zip(xs, ps):                                                        
        out.append(f'<circle class="ui-pt" cx="{sx(x):.1f}" cy="{sy(p):.1f}" r="4" fill="#d64545"><title>{feature} = {x:g}: {CLASS_NAMES[1]} {p * 100:.1f}%</title></circle>')
    cx = sx(xs[np.abs(xs - current_x).argmin()]) if is_category else sx(min(max(current_x, lo), hi))  
    out.append(f'<line x1="{cx:.1f}" x2="{cx:.1f}" y1="{T}" y2="{T + ph}" stroke="#142b53" stroke-width="2"/>'
               f'<circle cx="{cx:.1f}" cy="{sy(current_p):.1f}" r="7" fill="#142b53" stroke="white" stroke-width="2"><title>Current: {feature} = {current_x:g}, {current_p * 100:.1f}%</title></circle>')
    step = max(1, len(xs) // 6)                                                    
    for i in range(0, len(xs), step):
        out.append(f'<text x="{sx(xs[i]):.1f}" y="{H - 26}" font-size="13" text-anchor="middle">{xs[i]:g}</text>')
    out.append(f'<text x="{L + pw / 2}" y="{H - 6}" font-size="13" text-anchor="middle">{feature} (grey = dataset ke patients)</text></svg>')
    return f'<div class="hc-chart"><h4>{title}</h4>{"".join(out)}</div>'


def percentile_text(value, population):
    """Slider value ka dataset me position."""
    pop = np.asarray(population, float)
    pct = (pop < value).mean() * 100
    return f"Dataset me {pct:.0f}% patients se zyada (median {np.median(pop):g})"

Overwriting components.py


In [19]:
import numpy as np, importlib, time
import components as ui; importlib.reload(ui)
from confidence import analyse, sweep, sweep_values, FEATURES
from cache import PredictionCache

df_all = pd.read_csv('heart.csv').drop_duplicates()
x = df_all[FEATURES].values[0].astype(float)
cache_t5 = PredictionCache(max_size=512, ttl=600)
age_i = FEATURES.index('age')

print('age  P(Higher Risk)  time')
last = None
for age in range(40, 71, 5):
    xx = x.copy(); xx[age_i] = age
    t = time.perf_counter()
    p = cache_t5.get_or_compute(('t5', tuple(xx)), lambda: analyse(model, xx))['probs'][1]
    print(f'{age}   {p:6.1%}        {(time.perf_counter() - t) * 1000:.1f} ms')

# Curve ka har point model se match karta hai?
vals = sweep_values(rules['age'], 25); curve = sweep(model, x, 'age', vals)
direct = [model.predict_proba(pd.DataFrame([np.where(np.arange(13) == age_i, v, x)], columns=FEATURES))[0, 1] for v in vals]
print('curve vs model.predict_proba, max difference =', float(np.abs(np.array(curve) - np.array(direct)).max()))

age  P(Higher Risk)  time
40    74.0%        13.3 ms
45    73.0%        11.7 ms
50    74.0%        8.1 ms
55    78.0%        5.9 ms
60    73.0%        5.9 ms
65    78.0%        5.4 ms
70    76.0%        4.1 ms
curve vs model.predict_proba, max difference = 0.0


In [20]:
from IPython.display import HTML
from confidence import CSS
p = analyse(model, x)['probs'][1]
p_prev = 0.35                       
vals_a = sweep_values(rules['age'], 25); ps_a = sweep(model, x, 'age', vals_a)
vals_c = sweep_values(rules['cp']);       ps_c = sweep(model, x, 'cp', vals_c)
demo_html = (CSS + ui.CSS + ui.risk_card(p, 0.5, p_prev, ['chol'])
             + ui.curve_chart('Risk curve: age badalne par risk', 'age', vals_a, ps_a, float(x[0]), p, df_all['age'].values, False)
             + ui.curve_chart('Risk curve: cp (category)', 'cp', vals_c, ps_c, float(x[2]), p, df_all['cp'].values, True))
print(ui.percentile_text(63, df_all['age'].values))
HTML(demo_html)

Dataset me 80% patients se zyada (median 55.5)


In [21]:
import html as _h
from IPython.display import HTML
page = '<html><head><meta name="viewport" content="width=device-width,initial-scale=1"></head><body style="margin:0;background:#f3f6fb;font-family:sans-serif"><div style="padding:12px 10px">' + demo_html + '</div></body></html>'
frames = ''.join(f'<div style="margin:8px;display:inline-block;vertical-align:top"><b>{n} ({w}px)</b><br>'
                 f'<iframe srcdoc="{_h.escape(page)}" style="width:{w}px;height:{hh}px;border:1px solid #aaa;background:white"></iframe></div>'
                 for n, w, hh in [('Phone', 360, 700), ('Tablet', 768, 700)])
HTML(frames)

In [22]:
import os
open('responsive_test.html', 'w').write(page.replace('<div style="padding:12px 10px">', '<div style="max-width:850px;margin:auto;padding:12px 10px">'))
try:
    from playwright.sync_api import sync_playwright
    with sync_playwright() as pw:
        b = pw.chromium.launch()
        for name, w in [('phone', 360), ('tablet', 768), ('laptop', 1280)]:
            pg = b.new_context(viewport={'width': w, 'height': 800}).new_page()
            pg.goto('file://' + os.path.abspath('responsive_test.html')); pg.wait_for_timeout(1200)
            r = pg.evaluate("() => { const d = document.documentElement; const t = [...document.querySelectorAll('svg text')].map(e => e.getBoundingClientRect().height);"
                            " return {scrollW: d.scrollWidth, clientW: d.clientWidth, minSvgTextPx: Math.round(Math.min(...t))} }")
            print(f'{name:6s} {w:5d}px  sideways scroll: {r["scrollW"] > r["clientW"]}  | smallest chart text: {r["minSvgTextPx"]}px')
        b.close()
except Exception as e:
    print('Playwright is machine par nahi hai (optional):', type(e).__name__, '- upar iframe preview se visually dekho.')

Playwright is machine par nahi hai (optional): ModuleNotFoundError - upar iframe preview se visually dekho.


## Task 6: Display SHAP values in the UI for interpretability by manually computing feature attributions, developing an interactive visualization that allows users to explore individual contributions, and ensuring transparency in model decisions.

In [23]:
%%writefile shap_values.py
"""
HeartCare - SHAP values (haath se) aur unka interactive display.

Shapley value: ek feature ka "fair share" jo prediction ko average patient se door le jata hai.
  phi_i = sum over coalitions S (i ke bina)  |S|! (M-|S|-1)! / M!  *  [ v(S + i) - v(S) ]
  v(S)  = S ke features patient se, baaki features background (reference) patients se; model ka average output.
Yahan koi `shap` library nahi. 13 features = 2^13 = 8192 coalitions, saare exactly evaluate hote hain.
Contributions "probability points" (percentage points) me hain: P(class 1) = P(Higher Risk).
"""
from math import factorial
import numpy as np
import pandas as pd

from confidence import FEATURES, CLASS_NAMES, COLORS

M = len(FEATURES)
MASKS = ((np.arange(2 ** M)[:, None] >> np.arange(M)) & 1).astype(bool)   
SIZES = MASKS.sum(axis=1)
WEIGHTS = np.array([factorial(s) * factorial(M - s - 1) / factorial(M) for s in range(M)])


def build_background(df, k=20, seed=0):
    """Reference patients: training data se k random rows."""
    return df[FEATURES].sample(k, random_state=seed).to_numpy(dtype=float)


def coalition_values(model, x, background):
    """v(S) har coalition ke liye (8192 values): model ka average P(class 1)."""
    k = len(background)
    X = np.where(MASKS[None, :, :], np.asarray(x, float)[None, None, :], background[:, None, :])   
    p = model.predict_proba(pd.DataFrame(X.reshape(-1, M), columns=FEATURES))[:, 1]
    return p.reshape(k, -1).mean(axis=0)


def shapley_exact(model, x, background):
    """Exact Shapley values. Return: phi (13,), base value, prediction f(x)."""
    v = coalition_values(model, x, background)
    phi = np.zeros(M)
    for i in range(M):
        without = np.where(~MASKS[:, i])[0]                    
        phi[i] = (WEIGHTS[SIZES[without]] * (v[without | (1 << i)] - v[without])).sum()
    return phi, float(v[0]), float(v[-1])                      


def shapley_sampling(model, x, background, n_perm=200, seed=0):
    """Doosra independent tareeka (verification ke liye): random orderings me feature ek-ek karke patient wali value par laate hain."""
    rng = np.random.default_rng(seed)
    x = np.asarray(x, float)
    phi = np.zeros(M)
    for _ in range(n_perm):
        order = rng.permutation(M)
        b = background[rng.integers(len(background))].copy()
        rows = [b.copy()]
        for j in order:
            b[j] = x[j]; rows.append(b.copy())
        p = model.predict_proba(pd.DataFrame(np.array(rows), columns=FEATURES))[:, 1]
        phi[order] += np.diff(p)
    return phi / n_perm


def explain(model, x, background):
    phi, base, fx = shapley_exact(model, x, background)
    return {"phi": phi, "base": base, "pred": fx, "x": np.asarray(x, float), "k": len(background)}


def fmt(v):
    return f"{v:g}"


def plain_summary(info, filled=()):
    """Top drivers ek line me, aam bhasha me."""
    order = np.argsort(-np.abs(info["phi"]))[:3]
    parts = []
    for i in order:
        f = FEATURES[i]; pts = info["phi"][i] * 100
        direction = CLASS_NAMES[1] if pts > 0 else CLASS_NAMES[0]
        est = " (estimated)" if f in filled else ""
        parts.append(f"{f} = {fmt(info['x'][i])}{est} {direction} ki taraf {abs(pts):.1f} pts")
    return "Sabse bade kaaran: " + "; ".join(parts) + "."


def feature_detail(info, feature, bg_mean, filled=()):
    """Ek feature ka detail: value, average patient se compare, contribution."""
    i = FEATURES.index(feature); pts = info["phi"][i] * 100
    side = f"{CLASS_NAMES[1]} ki taraf" if pts > 0 else f"{CLASS_NAMES[0]} ki taraf"
    est = " Ye value aapne nahi di, isliye typical value se bhari gayi hai." if feature in filled else ""
    return (f"<b>{feature} = {fmt(info['x'][i])}</b> (reference patients ka average {bg_mean[i]:.1f}). "
            f"Is feature ne prediction ko <b>{abs(pts):.1f} percentage points {side}</b> dhakela.{est}")


SHAP_CSS = """
<style>
.sv-row { display: flex; align-items: center; margin: 6px 0; padding: 2px 4px; border-radius: 6px; }
.sv-row.sel { background: #eef4ff; outline: 2px solid #244577; }
.sv-label { width: 128px; font-size: 14px; color: #334155; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }
.sv-track { position: relative; flex: 1; height: 22px; background: #f1f5f9; border-radius: 6px; }
.sv-mid { position: absolute; left: 50%; top: 0; bottom: 0; width: 2px; background: #94a3b8; }
.sv-bar { position: absolute; top: 3px; bottom: 3px; width: var(--to); animation: sv-grow 0.9s ease-out; }
.sv-bar.pos { left: 50%; border-radius: 0 5px 5px 0; }
.sv-bar.neg { right: 50%; border-radius: 5px 0 0 5px; }
.sv-val { width: 78px; text-align: right; font-size: 13px; font-weight: 600; color: #142b53; }
.sv-total { display: flex; justify-content: space-between; font-size: 14px; padding: 8px 4px 0 4px; color: #142b53; border-top: 1px solid #e2e8f0; margin-top: 8px; flex-wrap: wrap; gap: 6px; }
.sv-legend { font-size: 12px; color: #475569; margin-top: 6px; }
@keyframes sv-grow { from { width: var(--from); } to { width: var(--to); } }
@media (max-width: 600px) { .sv-label { width: 92px; font-size: 12px; } .sv-val { width: 62px; font-size: 12px; } }
@media (prefers-reduced-motion: reduce) { .sv-bar { animation: none; } }
</style>
"""


def shap_chart(info, top_n=8, selected=None, prev=None, filled=()):
    """Diverging bars: kaun sa feature prediction ko kis taraf aur kitna dhakel raha hai.
    prev = {feature: pichli phi} (animation ka start)."""
    phi = info["phi"]; order = np.argsort(-np.abs(phi)); top = order[:top_n]; rest = order[top_n:]
    scale = max(np.abs(phi[order[0]]), 1e-9)                                   
    rows = ""
    for i in top:
        f = FEATURES[i]; v = phi[i]; cls = "pos" if v > 0 else "neg"
        old = (prev or {}).get(f)
        start = abs(old) / scale * 50 if old is not None and np.sign(old) == np.sign(v) else 0.0
        color = COLORS[1] if v > 0 else COLORS[0]
        est = "*" if f in filled else ""
        sel = " sel" if f == selected else ""
        rows += (f'<div class="sv-row{sel}" title="{f}: {v * 100:+.2f} percentage points"><div class="sv-label">{f} = {fmt(info["x"][i])}{est}</div>'
                 f'<div class="sv-track"><div class="sv-mid"></div><div class="sv-bar {cls}" style="--from:{start:.1f}%;--to:{abs(v) / scale * 50:.1f}%;background:{color}"></div></div>'
                 f'<div class="sv-val">{v * 100:+.1f} pts</div></div>')
    if len(rest):
        other = phi[rest].sum() * 100
        rows += (f'<div class="sv-row"><div class="sv-label">baaki {len(rest)} features</div><div class="sv-track"><div class="sv-mid"></div></div>'
                 f'<div class="sv-val">{other:+.1f} pts</div></div>')
    total = (f'<div class="sv-total"><span>Average patient: <b>{info["base"] * 100:.1f}%</b></span>'
             f'<span>+ contributions: <b>{phi.sum() * 100:+.1f} pts</b></span><span>= Is patient: <b>{info["pred"] * 100:.1f}%</b></span></div>')
    legend = (f'<div class="sv-legend"><span style="color:{COLORS[1]}">&#9632;</span> {CLASS_NAMES[1]} ki taraf &nbsp; '
              f'<span style="color:{COLORS[0]}">&#9632;</span> {CLASS_NAMES[0]} ki taraf &nbsp; (* = estimated value)</div>')
    return f'<div class="hc-chart"><h4>Kis feature ne prediction ko kitna dhakela (SHAP values)</h4>{rows}{total}{legend}</div>'

Overwriting shap_values.py


In [24]:
import importlib, time, numpy as np
import shap_values as sv; importlib.reload(sv)

bg = sv.build_background(df_all, k=20)                     
errs, secs = [], []
for x_i in df_all[FEATURES].values[:30].astype(float):
    t = time.perf_counter(); info = sv.explain(model, x_i, bg); secs.append(time.perf_counter() - t)
    errs.append(abs(info['base'] + info['phi'].sum() - info['pred']))
print('30 patients | base + sum(SHAP) = prediction, max error:', max(errs))
print(f'time per patient: {np.median(secs) * 1000:.0f} ms (8192 coalitions)')
info = sv.explain(model, x, bg)
print(f"average patient {info['base']:.1%} + contributions {info['phi'].sum():+.1%} = {info['pred']:.1%}")

30 patients | base + sum(SHAP) = prediction, max error: 2.220446049250313e-16
time per patient: 1132 ms (8192 coalitions)
average patient 47.1% + contributions +30.8% = 78.0%


In [25]:
rows = []
for j in range(5):
    xj = df_all[FEATURES].values[j * 40].astype(float)
    ex = sv.explain(model, xj, bg)['phi']
    sm = sv.shapley_sampling(model, xj, bg, n_perm=200)
    rows.append({'patient': j * 40, 'correlation': round(float(np.corrcoef(ex, sm)[0, 1]), 3),
                 'max diff (pts)': round(float(np.abs(ex - sm).max() * 100), 2)})
pd.DataFrame(rows)

,patient,correlation,max diff (pts)
0,0,0.997,1.05
1,40,0.996,1.28
2,80,0.997,0.87
3,120,0.995,1.77
4,160,0.995,1.21


In [26]:
xj = df_all[FEATURES].values[0].astype(float)
phis = [sv.explain(model, xj, sv.build_background(df_all, k=20, seed=s))['phi'] for s in range(5)]
corr = np.corrcoef(phis)[np.triu_indices(5, 1)]
top3 = [set(np.argsort(-np.abs(p))[:3]) for p in phis]
overlap = [len(top3[a] & top3[b]) for a in range(5) for b in range(a + 1, 5)]
print(f'seed-to-seed correlation: min {corr.min():.2f}, mean {corr.mean():.2f}')
print(f'top-3 features common: min {min(overlap)} of 3, mean {np.mean(overlap):.1f} of 3')
print('top-3 (seed 0):', [FEATURES[i] for i in np.argsort(-np.abs(phis[0]))[:3]])

seed-to-seed correlation: min 0.93, mean 0.96
top-3 features common: min 2 of 3, mean 2.1 of 3
top-3 (seed 0): ['ca', 'cp', 'thal']


In [27]:
sample = df_all[FEATURES].astype(float).sample(60, random_state=1)
S = np.array([sv.explain(model, r.values, bg)['phi'] for _, r in sample.iterrows()]) * 100
tab = []
for f, cut in [('ca', 0.5), ('exang', 0.5), ('oldpeak', 1.0), ('thalach', 150), ('age', 55), ('sex', 0.5)]:
    i = FEATURES.index(f); hi = sample[f].values > cut
    tab.append({'feature': f, 'cut': cut, 'avg SHAP when value <= cut': round(float(S[~hi, i].mean()), 1),
                'avg SHAP when value > cut': round(float(S[hi, i].mean()), 1)})
print('Positive = Higher Risk (class 1) ki taraf')
pd.DataFrame(tab).set_index('feature')

Positive = Higher Risk (class 1) ki taraf


,cut,avg SHAP when value <= cut,avg SHAP when value > cut
feature,,,
ca,0.5,9.8,-6.8
exang,0.5,2.4,-5.2
oldpeak,1.0,4.9,-5.7
thalach,150.0,-5.4,3.9
age,55.0,1.9,-2.4
sex,0.5,7.3,-1.1


In [28]:
from IPython.display import HTML
from confidence import CSS
import shap_values as sv
info = sv.explain(model, x, bg)
print(sv.plain_summary(info, filled=['chol']))
HTML(CSS + sv.SHAP_CSS + sv.shap_chart(info, top_n=8, selected='ca', filled=['chol'])
     + '<p style="font-family:sans-serif;max-width:600px">' + sv.feature_detail(info, 'ca', bg.mean(axis=0), ['chol']) + '</p>')

Sabse bade kaaran: ca = 0 Higher Risk ki taraf 12.8 pts; cp = 3 Higher Risk ki taraf 11.1 pts; thal = 1 Higher Risk ki taraf 7.4 pts.


In [29]:
%%writefile app.py
import time
import joblib
import numpy as np
import pandas as pd
import streamlit as st

from input import FEATURES, load_rules, validate_input, predict_batch     
from cache import PredictionCache                                         
from confidence import (CSS, CLASS_NAMES, COLORS, analyse, decide, sweep, 
                        sweep_values, bar_chart, column_chart)
import components as ui                                                    
import shap_values as sv                                                  
from feedback import add_feedback                                        

st.set_page_config(
    page_title="HeartCare",
    layout="centered"
)

MODEL_FILE = "heart_model_v20261013_160903.joblib"
DATA_FILE = "heart.csv"
FEEDBACK_FILE = "feedback.csv"
MAX_BATCH_ROWS = 100000
MAX_UNKNOWN = 3


@st.cache_resource
def load_everything():
    model = joblib.load(MODEL_FILE)
    rules = load_rules(DATA_FILE)
    cache = PredictionCache(max_size=512, ttl=600)
    df = pd.read_csv(DATA_FILE).drop_duplicates()
    background = sv.build_background(df, k=20)
    return model, rules, cache, MODEL_FILE, df, background


try:
    model, rules, cache, model_version, data_df, background = load_everything()
except FileNotFoundError as e:
    st.error(f"File nahi mili: {e.filename}. app.py ke same folder me rakho.")
    st.stop()

SECTIONS = [
    ("Basic", ["age", "sex"]),
    ("Chest & heart rate", ["cp", "exang", "thalach", "oldpeak", "slope"]),
    ("Tests (blood / ECG)", ["trestbps", "chol", "fbs", "restecg"]),
    ("Scan results", ["ca", "thal"]),
]
LABELS = {"age": "Age", "sex": "Sex", "cp": "Chest Pain Type", "trestbps": "Resting Blood Pressure",
          "chol": "Cholesterol", "fbs": "Fasting Blood Sugar", "restecg": "Resting ECG",
          "thalach": "Maximum Heart Rate", "exang": "Exercise Induced Angina", "oldpeak": "Oldpeak",
          "slope": "Slope", "ca": "Major Vessels", "thal": "Thal"}
HELP = {"age": "Years me.", "sex": "Dataset ka code: 0 ya 1.", "cp": "Chest pain type ka code (0 se 3).",
        "trestbps": "Resting blood pressure, mm Hg.", "chol": "Serum cholesterol, mg/dl.",
        "fbs": "Fasting blood sugar ka code (0 ya 1).", "restecg": "Resting ECG result ka code (0 se 2).",
        "thalach": "Exercise me maximum heart rate, beats per minute.", "exang": "Exercise se chest pain hua? 0 ya 1.",
        "oldpeak": "Exercise ke baad ST depression (number).", "slope": "ST segment slope ka code (0 se 2).",
        "ca": "Major vessels ki ginti (dataset code 0 se 4).", "thal": "Thal test ka code (0 se 3)."}
WHOLE = {"age", "trestbps", "chol", "thalach"}       


def previous(key, current):
    """Chart ki pichli values (animation start). Session me yaad rakhte hain."""
    old = st.session_state.get("prev", {}).get(key)
    st.session_state.setdefault("prev", {})[key] = list(current)
    return old


def default_for(f):
    r = rules[f]
    if r["kind"] == "number":
        return int(round(r["fill"])) if f in WHOLE else float(r["fill"])
    return int(r["fill"])


def reset_inputs():
    for f in FEATURES:
        st.session_state["in_" + f] = default_for(f)
    st.session_state["unknown"] = []
    st.session_state.pop("check", None)


def widget(f):
    """Number -> slider, 2 se 4 options -> radio, zyada options -> selectbox. Default median/mode."""
    r = rules[f]; key = "in_" + f
    if r["kind"] == "number":
        if f in WHOLE:
            st.session_state.setdefault(key, default_for(f))
            v = st.slider(LABELS[f], int(r["min"]), int(r["max"]), key=key, help=HELP[f])
        else:
            st.session_state.setdefault(key, default_for(f))
            v = st.slider(LABELS[f], float(r["min"]), float(r["max"]), step=0.1, key=key, help=HELP[f])
        st.caption(ui.percentile_text(v, data_df[f].values))
        return v
    options = [int(a) for a in r["allowed"]]
    st.session_state.setdefault(key, default_for(f))
    if len(options) <= 4:
        return st.radio(LABELS[f], options, key=key, horizontal=True, help=HELP[f])
    return st.selectbox(LABELS[f], options, key=key, help=HELP[f])


st.markdown("""
<style>
.stApp {
    background: #f3f6fb;
}

header, footer {
    visibility: hidden;
}

.main .block-container {
    max-width: 850px;
    padding: 25px 20px;
}

.header {
    background: #142b53;
    color: white;
    padding: 25px;
    border-radius: 14px;
    text-align: center;
    margin-bottom: 25px;
}

.section {
    background: white;
    padding: 20px;
    border-radius: 12px;
    margin-bottom: 18px;
    border: 1px solid #e2e8f0;
}

.section h3 {
    color: #142b53;
}

.stButton button {
    background: #142b53;
    color: white;
    width: 100%;
    height: 45px;
    border-radius: 8px;
}

.stButton button:hover {
    background: #244577;
    color: white;
}

@media(max-width: 600px) {
    .main .block-container {
        padding: 12px 10px;
    }

    .header {
        padding: 18px;
    }

    .header h1 {
        font-size: 24px;
    }

    .section {
        padding: 14px;
    }
}
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="header">
    <h1>HeartCare</h1>
    <p>Heart Disease Risk Prediction System</p>
</div>
""", unsafe_allow_html=True)

st.markdown("""
<div class="section">
    <h3>Patient Information</h3>
    <p>Enter patient medical details below.</p>
</div>
""", unsafe_allow_html=True)


st.caption("Sliders/options se values chuno. Koi value pata na ho to neeche 'Values I don't know' me chuno (max 3), wo median/mode se bhar di jayegi.")

top = st.empty()
st.markdown(CSS + ui.CSS + sv.SHAP_CSS, unsafe_allow_html=True)

values = {}
for title, feats in SECTIONS:
    st.markdown(f'<div class="section"><h3>{title}</h3></div>', unsafe_allow_html=True)
    for f in feats:
        values[f] = widget(f)

unknown = st.multiselect("Values I don't know (max 3)", [LABELS[f] for f in FEATURES], key="unknown")
too_many = len(unknown) > MAX_UNKNOWN
if too_many:
    st.error(f"Maximum {MAX_UNKNOWN} values 'unknown' ho sakti hain. Ek-do hata do.")

live = st.checkbox("Live update (slider hilate hi prediction update ho)", value=True)
clicked = st.button("Predict Heart Disease Risk")
st.button("Reset to typical values", on_click=reset_inputs)

if (clicked or live) and not too_many:

    record = {f: (None if LABELS[f] in unknown else values[f]) for f in FEATURES}
    check = validate_input(record, rules)

    for message in check["errors"]:
        st.error(message)

    if check["ok"]:
        st.session_state["check"] = check       
    else:
        st.session_state.pop("check", None)

if too_many:
    st.session_state.pop("check", None)        

check = st.session_state.get("check")

if check:

    st.markdown("""
<div class="section">
    <h3>Prediction & Confidence</h3>
</div>
""", unsafe_allow_html=True)

    threshold = st.slider("Decision threshold (%): Higher Risk tab maana jaye jab probability isse zyada ho", 1, 99, 50, key="threshold") / 100
    explore = st.selectbox("Risk curve: ek feature badalke dekho", FEATURES, index=0, key="explore")
    top_n = st.slider("SHAP: kitne top features dikhayein", 3, 13, 8, key="top_n")

    start = time.perf_counter()
    key = (model_version, tuple(float(v) for v in check["x"]))
    info = cache.get_or_compute(key, lambda: analyse(model, check["x"]))
    elapsed = (time.perf_counter() - start) * 1000

    for message in check["warnings"]:
        st.warning(message)

    filled = list(check["filled"])
    probs = info["probs"]
    prediction = decide(probs, threshold)

    p1 = float(probs[1])
    prev_card = st.session_state.get("last_p1")
    st.session_state["last_p1"] = p1
    top.markdown(ui.risk_card(p1, threshold, prev_card, filled), unsafe_allow_html=True)

    st.write("Model Confidence:", round(max(probs) * 100, 2), "%")
    if threshold != 0.5:
        st.caption(f"Custom threshold {threshold * 100:.0f}%: {CLASS_NAMES[1]} probability {probs[1] * 100:.1f}% hai, "
                   f"isliye prediction {CLASS_NAMES[prediction]} dikh raha hai (model ka default threshold 50% hai).")
    st.caption(f"{info['votes'][1]} of {info['n_trees']} trees ne {CLASS_NAMES[1]} chuna | trees ke beech spread (std): {info['std']:.2f}")

    labels = [CLASS_NAMES[0], CLASS_NAMES[1]]
    st.markdown(bar_chart("Class probabilities (trees ka average)", labels, probs,
                          prev=previous("probs", probs), colors=[COLORS[0], COLORS[1]], threshold=threshold,
                          note=f"Kali line = threshold ({threshold * 100:.0f}%)"), unsafe_allow_html=True)

    votes = info["votes"] / info["n_trees"]
    st.markdown(bar_chart("Tree votes (kitni trees ne kaunsi class chuni)", labels, votes,
                          prev=previous("votes", votes), colors=[COLORS[0], COLORS[1]]), unsafe_allow_html=True)

    hist = info["hist"]
    bins = [f"{i * 10}-{i * 10 + 10}" for i in range(len(hist))]
    st.markdown(column_chart(f"Har tree ka {CLASS_NAMES[1]} probability (%) - kitni trees kis range me",
                             bins, hist / hist.max(), prev=previous("hist", hist / hist.max()),
                             color="#244577", counts=[int(h) for h in hist],
                             note="Bar ke upar tree ginti."), unsafe_allow_html=True)

    x = check["x"]; r = rules[explore]; fi = FEATURES.index(explore)
    xs = sweep_values(r, 25) if r["kind"] == "number" else sweep_values(r)
    ps = sweep(model, x, explore, xs)
    st.markdown(ui.curve_chart(f"Risk curve: {explore} badalne par risk", explore, xs, ps, float(x[fi]), p1,
                               data_df[explore].values, r["kind"] == "category", threshold), unsafe_allow_html=True)

    st.markdown("""
<div class="section">
    <h3>Why this prediction? (SHAP values)</h3>
    <p>Har feature ne risk ko average patient se kitna upar/neeche dhakela. Ye Shapley values
    haath se nikali gayi hain (13 features ke saare combinations).</p>
</div>
""", unsafe_allow_html=True)

    skey = (model_version, "shap", tuple(float(v) for v in x), len(background))
    sinfo = cache.get_or_compute(skey, lambda: sv.explain(model, x, background))
    inspect = st.selectbox("Ek feature ko detail me dekho", ["(none)"] + FEATURES, key="inspect")
    sel = None if inspect == "(none)" else inspect

    old_phi = previous("shap", sinfo["phi"])
    shap_prev = dict(zip(FEATURES, old_phi)) if old_phi else None

    st.caption(sv.plain_summary(sinfo, filled))
    st.markdown(sv.shap_chart(sinfo, top_n, sel, prev=shap_prev, filled=filled), unsafe_allow_html=True)
    if sel:
        st.markdown(sv.feature_detail(sinfo, sel, background.mean(axis=0), filled), unsafe_allow_html=True)
    st.caption("Transparency: ye model ke patterns hain, medical karan (cause) nahi. Reference = dataset ke 20 patients ka average. "
               "Base + saare contributions = final probability (exactly).")

    if not live:
        st.caption("Ye result pichhle Predict click ke inputs ka hai. Naye inputs ke liye dobara Predict dabao ya Live update on karo.")

    stats = cache.stats()
    st.caption(f"Is request me {elapsed:.2f} ms | Cache: {stats['hits']} hits, {stats['misses']} misses, "
               f"{stats['waits']} coalesced, hit rate {stats['hit_rate_%']}% (size {stats['size']})")

st.markdown("""
<div class="section">
    <h3>Batch Prediction (CSV)</h3>
    <p>Bahut patients ek saath: CSV me columns age, sex, cp, trestbps, chol, fbs,
    restecg, thalach, exang, oldpeak, slope, ca, thal hone chahiye.</p>
</div>
""", unsafe_allow_html=True)

uploaded = st.file_uploader("Upload CSV", type=["csv"])

data = None

if uploaded is not None:

    try:
        data = pd.read_csv(uploaded)
    except Exception:
        st.error("CSV padh nahi paye. File sahi CSV format me honi chahiye.")

if data is not None:

    clean_names = [str(c).strip().lower().replace(" ", "_") for c in data.columns]
    missing_cols = [f for f in FEATURES if f not in clean_names]

    if len(data) == 0:
        st.error("CSV khali hai.")
    elif len(data) > MAX_BATCH_ROWS:
        st.error(f"Max {MAX_BATCH_ROWS} rows allowed hain.")
    else:
        if missing_cols:
            st.warning(f"Ye columns CSV me nahi mile (missing maane gaye): {missing_cols}")

        start = time.perf_counter()
        results = predict_batch(model, rules, data)
        elapsed = (time.perf_counter() - start) * 1000

        rejected = int((results["status"] == "rejected").sum())
        st.write(f"Rows: {len(data)} | Predicted: {len(data) - rejected} | Rejected: {rejected}")
        st.caption(f"{elapsed:.0f} ms ({elapsed / len(data):.3f} ms per row)")

        output = pd.concat([data, results], axis=1)
        st.dataframe(output.head(100))

        st.download_button("Download results", output.to_csv(index=False), "heart_predictions.csv", "text/csv")


st.markdown("""
<div class="section">
    <h3>Feedback</h3>
    <p>Ye app kaisa laga? Isse hum interface sudharte hain.</p>
</div>
""", unsafe_allow_html=True)

fb_device = st.radio("Aap kis device par ho?", ["Phone", "Tablet", "Laptop"], horizontal=True, key="fb_device")
fb_task = st.radio("Kya aap apna kaam (risk dekhna) bina madad ke kar paye?", ["Yes", "Partly", "No"], horizontal=True, key="fb_task")
fb_ease = st.slider("Use karna kitna aasaan laga? (1 = mushkil, 5 = bahut aasaan)", 1, 5, 4, key="fb_ease")
fb_text = st.text_area("Kuch atka ya confusing laga? (optional)", key="fb_text", max_chars=500)
if st.button("Send feedback"):
    try:
        add_feedback(FEEDBACK_FILE, fb_device, fb_task, fb_ease, fb_text)
        st.success("Shukriya! Feedback save ho gaya.")
    except Exception as e:
        st.error(f"Feedback save nahi hua: {e}")

st.markdown("""
<div style="text-align:center;color:#64748b;padding:20px;">
    HeartCare | Educational Purpose Only
    <br>
    Not a substitute for professional medical diagnosis.
</div>
""", unsafe_allow_html=True)

Overwriting app.py


In [30]:
import sys, types, time, warnings; warnings.filterwarnings('ignore')

def ui_probe(app_file, session=None, typed=None, clicks=(), resources=None):
    """Return: widgets ki ginti + kya result dikha. session = v2 ke widget keys; typed = v1 ke text boxes."""
    session, typed, resources = session if session is not None else {}, typed or {}, resources if resources is not None else {}
    count, log = {}, []
    def tick(name): count[name] = count.get(name, 0) + 1
    st = types.ModuleType('streamlit')
    def cache_resource(f):
        def w(*a, **k):
            if f.__name__ not in resources: resources[f.__name__] = f(*a, **k)
            return resources[f.__name__]
        return w
    st.cache_resource = cache_resource; st.set_page_config = lambda **k: None
    st.markdown = lambda t, **k: log.append(('html', t)); st.caption = lambda m: log.append(('caption', m))
    st.write = lambda *a: log.append(('write', ' '.join(map(str, a))))
    st.error = lambda m: log.append(('error', m)); st.warning = lambda m: log.append(('warning', m)); st.success = lambda m: log.append(('success', m))
    st.dataframe = lambda d: None; st.download_button = lambda *a, **k: None; st.file_uploader = lambda *a, **k: None
    st.stop = lambda: (_ for _ in ()).throw(SystemExit)
    st.empty = lambda: types.SimpleNamespace(markdown=lambda t, **k: log.append(('html', t)))
    st.session_state = session
    def text_input(label, value=''): tick('text_input'); return typed.get(label.split(' (')[0], value)
    def selectbox(label, options, index=0, key=None, format_func=str, help=None):
        tick('selectbox'); return session.setdefault(key, options[index]) if key else options[index]
    def slider(label, lo, hi, v=None, step=None, key=None, help=None):
        tick('slider'); return session.setdefault(key, v) if key else v
    def radio(label, options, key=None, horizontal=False, help=None): tick('radio'); return session.setdefault(key, options[0])
    def multiselect(label, options, key=None): tick('multiselect'); return session.setdefault(key, [])
    def checkbox(label, value=False, key=None): tick('checkbox'); return session.setdefault(key or label, value)
    def button(label, on_click=None):
        tick('button')
        if label in clicks:
            if on_click: on_click()
            return True
        return False
    def text_area(label, key=None, max_chars=None): tick('text_area'); return session.setdefault(key, '')
    st.text_input, st.selectbox, st.slider, st.radio, st.multiselect, st.checkbox, st.button, st.text_area = \
        text_input, selectbox, slider, radio, multiselect, checkbox, button, text_area
    sys.modules['streamlit'] = st
    for m in ['input', 'cache', 'confidence', 'components', 'shap_values', 'feedback']: sys.modules.pop(m, None)
    t0 = time.perf_counter()
    try: exec(compile(open(app_file).read(), app_file, 'exec'), {'__name__': '__main__'})
    except SystemExit: pass
    ms = (time.perf_counter() - t0) * 1000
    shown = any(k == 'write' and 'Model Confidence' in v for k, v in log)
    return {'widgets': count, 'result_shown': shown, 'errors': [v for k, v in log if k == 'error'], 'ms': ms}

In [31]:
import pandas as pd
rows = []
res1, res2 = {}, {}
for name, f in [('v1', 'app_v1.py'), ('v2', 'app.py')]:
    res = res1 if name == 'v1' else res2
    r0 = ui_probe(f, resources=res)                                                
    if name == 'v1':
        r1 = ui_probe(f, typed={'Age': '63'}, resources=res)                        
        r2 = ui_probe(f, typed={'Age': 'abc'}, clicks=('Predict Heart Disease Risk',), resources=res)  
        bad = len(r2['errors']) > 0
    else:
        r1 = ui_probe(f, session={'in_age': 63}, resources=res)
        bad = False                                                                 
    w = r0['widgets']
    rows.append({'version': name,
                 'widgets': sum(w.values()),
                 'free-typing boxes': w.get('text_input', 0),
                 'result on open (no click)': r0['result_shown'],
                 'result after changing Age (no click)': r1['result_shown'],
                 'bad typed value can reach model': 'blocked by message' if bad else ('never possible' if name == 'v2' else '-')})
pd.DataFrame(rows).set_index('version').T

version,v1,v2
widgets,15,26
free-typing boxes,5,0
result on open (no click),False,True
result after changing Age (no click),False,True
bad typed value can reach model,blocked by message,never possible


In [32]:
import numpy as np
res = {}
ui_probe('app.py', resources=res)                                    
times_new, times_same = [], []
for age in [40, 45, 50, 55, 60, 65, 70]:
    times_new.append(ui_probe('app.py', session={'in_age': age}, resources=res)['ms'])     
for _ in range(7):
    times_same.append(ui_probe('app.py', session={'in_age': 70}, resources=res)['ms'])    
print(f'naya input par poora script (Streamlit ke bina): median {np.median(times_new):.0f} ms | wahi input: median {np.median(times_same):.0f} ms')

naya input par poora script (Streamlit ke bina): median 1217 ms | wahi input: median 48 ms


In [ ]:
!streamlit run app.py